In [153]:
import os
os.environ["KERAS_BACKEND"]="torch"
import glob
import io
import shutil
import zipfile
import numpy as np
import h5py
from PIL import Image
import keras
from keras import layers
from keras.applications.inception_v3 import InceptionV3, preprocess_input
from sklearn.model_selection import train_test_split

# Chemins et imports
Le notebook sélectionne le backend Torch avant d’importer Keras. Il cherche le dossier models à partir du répertoire de travail, puis définit les chemins des modèles et des données.

In [154]:
PROJECT_DIR = os.path.abspath(os.getcwd())
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
DATASETS_DIR = os.path.join(PROJECT_DIR, "Leyenda", "datasets")
CLASSIFIER_WEIGHTS = os.path.join(MODELS_DIR, "model_full.weights.h5")
DENOISER_MODEL = os.path.join(MODELS_DIR, "unet_denoiser_livrable2.keras")
PHOTOS_DIR = os.path.join(DATASETS_DIR, "photos")
DENOISED_DIR = os.path.join(DATASETS_DIR, "denoised_images")
COCO_DIR = os.path.join(DATASETS_DIR, "train2014")


# Etape 1 : classification Photo / Non-photo
Le modèle du livrable 1 reconnaît cinq classes : Painting, Photo, Schematics, Sketch et Text. Il faut donc garder ses cinq sorties : l’étape est dite binaire parce qu’on transforme ensuite sa prédiction en deux groupes, Photo et Non-photo.

Pour chaque image, le code :

la redimensionne en 128 × 128 ;
la passe au modèle, qui divise ses pixels par 255 grâce à sa couche Rescaling ;
récupère la probabilité associée à Photo ;
la compare au seuil 0,2301, défini dans le livrable 1.
Si la probabilité atteint le seuil, l’image est considérée comme une photo. Sinon, elle reste dans son dossier d’origine. Le code déplace les photos retenues dans datasets/photos — il ne les copie pas — mais ne déplace pas et ne supprime pas les autres images.

In [155]:
CLASS_NAMES = ["Painting", "Photo", "Schematics", "Sketch", "Text"]
NUM_CLASSES = len(CLASS_NAMES)
PHOTO_CLASS_INDEX = CLASS_NAMES.index("Photo")
THRESHOLD = 0.2301

In [156]:
def build_classifier_model(input_shape=(128, 128, 3), num_classes=NUM_CLASSES):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1./255),
        layers.Conv2D(32, (3, 3), activation="relu", name="conv2d_12"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", name="conv2d_13"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu", name="conv2d_14"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu", name="dense_8"),
        layers.Dense(num_classes, activation="softmax", name="dense_9"),
    ])
    # Le fichier est un modèle complet sauvegardé en HDF5 ; on charge directement ses poids.
    with h5py.File(CLASSIFIER_WEIGHTS, "r") as checkpoint:
        saved_layers = checkpoint["model_weights"]
        for layer in model.layers:
            if not layer.weights:
                continue
            saved_layer = saved_layers[layer.name]
            weight_names = saved_layer.attrs.get("weight_names", [])
            layer_weights = [saved_layer[name][()] for name in weight_names]
            layer.set_weights(layer_weights)

    return model


def load_image_as_array(path, target_size):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32)


def classify_images(image_paths, classifier, batch_size=32):
    results = []
    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        batch = np.stack([load_image_as_array(path, (128, 128)) for path in batch_paths])
        probabilities = classifier.predict(batch, verbose=0)
        for path, probas in zip(batch_paths, probabilities):
            proba_photo = float(probas[PHOTO_CLASS_INDEX])
            results.append((path, proba_photo, proba_photo >= THRESHOLD))
    return results

def move_photos_to_folder(classification_results, dest_dir=PHOTOS_DIR):
    os.makedirs(dest_dir, exist_ok=True)
    moved = []
    for path, proba, is_photo in classification_results:
        if is_photo:
            if os.path.abspath(os.path.dirname(path)) != os.path.abspath(dest_dir):
                filename = os.path.basename(path)
                dest_path = os.path.join(dest_dir, filename)
                if os.path.exists(dest_path):
                    base, extension = os.path.splitext(filename)
                    suffix = 1
                    while os.path.exists(dest_path):
                        dest_path = os.path.join(dest_dir, f"{base}_{suffix}{extension}")
                        suffix += 1
                shutil.move(path, dest_path)
                path = dest_path
            moved.append(path)
    return moved

# Etape 2 : débruitage avec le U-Net du livrable 2
Chaque photo retenue est chargée en RGB, redimensionnée en 299 × 299 et convertie en valeurs entre 0 et 1. Le U-Net du livrable 2 est ensuite appliqué à l’image, et le résultat est sauvegardé dans datasets/denoised_images.

Le U-Net effectue des opérations de pooling qui nécessitent des dimensions compatibles avec des réductions par deux. Le code complète donc temporairement les dimensions de 299 à 300, puis recadre le résultat à 299 × 299 avant de le sauvegarder.

In [157]:
def load_denoiser_model(model_path=DENOISER_MODEL):
    inputs = keras.Input(shape=(None, None, 3))

    e1 = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv2d")(inputs)
    e1 = layers.BatchNormalization(name="batch_normalization")(e1)
    e1 = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv2d_1")(e1)
    e1 = layers.BatchNormalization(name="batch_normalization_1")(e1)
    p1 = layers.MaxPooling2D((2, 2))(e1)

    e2 = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2d_2")(p1)
    e2 = layers.BatchNormalization(name="batch_normalization_2")(e2)
    e2 = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2d_3")(e2)
    e2 = layers.BatchNormalization(name="batch_normalization_3")(e2)
    p2 = layers.MaxPooling2D((2, 2))(e2)

    bottleneck = layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="conv2d_4")(p2)
    bottleneck = layers.BatchNormalization(name="batch_normalization_4")(bottleneck)
    bottleneck = layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="conv2d_5")(bottleneck)
    bottleneck = layers.BatchNormalization(name="batch_normalization_5")(bottleneck)

    d2 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding="same", name="conv2d_transpose")(bottleneck)
    d2 = layers.Concatenate()([d2, e2])
    d2 = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2d_6")(d2)
    d2 = layers.BatchNormalization(name="batch_normalization_6")(d2)
    d2 = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2d_7")(d2)
    d2 = layers.BatchNormalization(name="batch_normalization_7")(d2)

    d1 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding="same", name="conv2d_transpose_1")(d2)
    d1 = layers.Concatenate()([d1, e1])
    d1 = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv2d_8")(d1)
    d1 = layers.BatchNormalization(name="batch_normalization_8")(d1)
    d1 = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv2d_9")(d1)
    d1 = layers.BatchNormalization(name="batch_normalization_9")(d1)
    outputs = layers.Conv2D(3, (1, 1), activation="sigmoid", padding="same", name="conv2d_10")(d1)
    denoiser = keras.Model(inputs, outputs, name="UNet_Denoiser")

    with zipfile.ZipFile(model_path) as archive:
        weights_data = archive.read("model.weights.h5")
    with h5py.File(io.BytesIO(weights_data), "r") as checkpoint:
        saved_layers = checkpoint["layers"]
        for layer in denoiser.layers:
            if layer.weights:
                saved_vars = saved_layers[layer.name]["vars"]
                layer.set_weights([saved_vars[str(i)][()] for i in range(len(layer.weights))])
    return denoiser

def load_image_for_denoiser(path, target_size=(299, 299)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32) / 255.0

def denoise_images(image_paths, denoiser, output_dir=DENOISED_DIR, target_size=(299, 299)):
    os.makedirs(output_dir, exist_ok=True)
    denoised_paths = []

    for path in image_paths:
        arr = load_image_for_denoiser(path, target_size)
        height, width = arr.shape[:2]
        pad_height = (-height) % 4
        pad_width = (-width) % 4
        padded = np.pad(arr, ((0, pad_height), (0, pad_width), (0, 0)), mode="edge")
        batch = np.expand_dims(padded, axis=0)
        denoised = keras.ops.convert_to_numpy(denoiser(batch, training=False))[0]
        denoised = denoised[:height, :width, :]
        denoised = np.clip(denoised, 0.0, 1.0)

        denoised_img = Image.fromarray(np.rint(denoised * 255).astype("uint8"))
        filename = os.path.basename(path)
        out_path = os.path.join(output_dir, filename)
        denoised_img.save(out_path)
        denoised_paths.append(out_path)

    return denoised_paths


# Etape 3 : caractéristiques InceptionV3 sauvegardées en .npy
Les images débruitées sont prétraitées pour InceptionV3, puis transformées en vecteurs de caractéristiques. InceptionV3 est utilisée sans sa tête de classification ImageNet ; le vecteur obtenu résume l’image et pourra être utilisé ensuite par le modèle de génération de légendes.

Chaque vecteur est sauvegardé en fichier .npy, dans le même dossier que les images débruitées.

In [158]:
def build_inception_feature_extractor():
    # Charge InceptionV3 pré-entraîné sur ImageNet, sans la tête de classification
    return InceptionV3(weights="imagenet", include_top=False, pooling="avg")

def load_image_for_inception(path, target_size=(299, 299)):
    # Charge et prétraite une image pour InceptionV3 (preprocess_input fait la normalisation)
    img = Image.open(path).convert("RGB").resize(target_size)
    arr = np.array(img).astype("float32")
    arr = preprocess_input(arr)
    return arr

def extract_and_save_inception_features(image_paths, feature_extractor, output_dir=DENOISED_DIR, batch_size=8):
    os.makedirs(output_dir, exist_ok=True)
    npy_paths = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        batch_imgs = np.stack([load_image_for_inception(p) for p in batch_paths])

        bf = feature_extractor(batch_imgs, training=False)
        bf = keras.ops.convert_to_numpy(bf)

        for path, features in zip(batch_paths, bf):
            filename = os.path.splitext(os.path.basename(path))[0]
            npy_path = os.path.join(output_dir, filename + ".npy")
            np.save(npy_path, features)
            npy_paths.append(npy_path)

    return npy_paths



# Etape 4 : séparation train / validation / test (80/10/10)
Le code répartit les chemins des fichiers .npy entre entraînement, validation et test, avec une proportion visée de 80/10/10. Pour l’échantillon de test, cela a donné 9 / 1 / 1, car on ne peut pas obtenir exactement 80/10/10 avec seulement 11 éléments tout en gardant des fichiers dans chaque groupe.

À noter : le split crée des listes de chemins en mémoire ; il ne déplace pas les fichiers dans trois dossiers séparés et ne sauvegarde pas leurs étiquettes.

In [159]:
def split_dataset(image_paths):
    test_count = max(1, round(len(image_paths) * 0.1))
    val_count = max(1, round(len(image_paths) * 0.1))
    train_val_paths, test_paths = train_test_split(image_paths, test_size=test_count, random_state=42)
    train_paths, val_paths = train_test_split(train_val_paths, test_size=val_count, random_state=42)
    return train_paths, val_paths, test_paths

# Etape 5 : pipeline complète

In [160]:
def run_pipeline(image_paths):
    print(f"1. Classification de {len(image_paths)} images")
    classifier = build_classifier_model()
    classification_results = classify_images(image_paths, classifier)
    photos_moved = move_photos_to_folder(classification_results)
    print(f"{len(photos_moved)} photos déplacées dans {PHOTOS_DIR}.")
    print("2. Débruitage")
    denoiser = load_denoiser_model()
    denoised_paths = denoise_images(photos_moved, denoiser, output_dir=DENOISED_DIR)
    print(f"{len(denoised_paths)} images débruitées -> {DENOISED_DIR}")

    print("3. Extraction des caractéristiques InceptionV3")
    feature_extractor = build_inception_feature_extractor()
    npy_paths = extract_and_save_inception_features(denoised_paths, feature_extractor, output_dir=DENOISED_DIR)
    print(f"{len(npy_paths)} fichiers .npy -> {DENOISED_DIR}")

    print("4. Séparation train/validation/test")
    train_paths, val_paths, test_paths = split_dataset(npy_paths)
    print(f"Train : {len(train_paths)} | Validation : {len(val_paths)} | Test : {len(test_paths)}")

    return {
        "classification_results": classification_results,
        "photos_paths": photos_moved,
        "denoised_paths": denoised_paths,
        "npy_paths": npy_paths,
        "train_paths": train_paths,
        "val_paths": val_paths,
        "test_paths": test_paths,
    }


# ETAPE 6 : Test 

In [161]:
if __name__ == "__main__":
    N_TEST_IMAGES = 11  # Nombre d'images COCO utilisées pour le test.
    all_images = sorted(glob.glob(os.path.join(COCO_DIR, "*.jpg")))
    sample_size = min(N_TEST_IMAGES, len(all_images))
    sample_images = np.random.default_rng().choice(all_images, size=sample_size, replace=False).tolist()

    print(f"Test sur {len(sample_images)} images COCO...")
    results = run_pipeline(sample_images)
    print("Pipeline testée avec succès")

Test sur 11 images COCO...
1. Classification de 11 images
10 photos déplacées dans c:\Users\willi\Documents\Projet_Leyenda\Leyenda\datasets\photos.
2. Débruitage
10 images débruitées -> c:\Users\willi\Documents\Projet_Leyenda\Leyenda\datasets\denoised_images
3. Extraction des caractéristiques InceptionV3
10 fichiers .npy -> c:\Users\willi\Documents\Projet_Leyenda\Leyenda\datasets\denoised_images
4. Séparation train/validation/test
Train : 8 | Validation : 1 | Test : 1
Pipeline testée avec succès
